# DSSATLab v0.1 walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_walkthrough.ipynb)

This notebook exercises everything v0.1 does, in the order it happens:

| Step | What we test |
|---|---|
| 1 | Install the package and look at the environment (`detect()`) |
| 2 | Discovery when nothing is installed (`connect()` errors) |
| 3 | Error messages for bad input |
| 4 | Build DSSAT from source (`install()`), several minutes on Colab |
| 5 | What was written to disk (cache, manifest, config) |
| 6 | Discovery after install: saved config, `DSSAT_HOME`, `PATH`, managed cache |
| 7 | Cache reuse (second `install()` is instant) |
| 8 | Explicit `connect(executable=...)` / `connect(path=...)` |
| 9 | Failure path: a version that does not exist |
| 10 | Reset |

Run the cells top to bottom (**Runtime > Run all**, or step by step).

## 1. Install the package and inspect the environment

In [ ]:
# Latest code from GitHub. To test the released package instead: %pip install -q dssatlab
%pip install -q "git+https://github.com/AbdelrahmanAmr3/dssatlab.git"

In [ ]:
import dssatlab as dl
print("dssatlab", dl.__version__)
print("public API:", dl.__all__)

In [ ]:
# detect() is read-only: it never installs, downloads, or writes anything.
info = dl.detect()
print(info)
assert info.os_name == "linux"
print("Running on Colab:", info.is_colab)
print("DSSAT already found:", info.dssat_executable)

## 2. Discovery with nothing installed

`connect(interactive=False)` checks, in order: saved config, `DSSAT_HOME`, `PATH` (`dscsm048`), then the managed cache.
On a fresh Colab runtime none exist, so we expect `DSSATNotFoundError` saying what was checked and what to do next.

In [ ]:
import os
from dssatlab import config

print("config file:", config.config_file(), "| exists:", config.config_file().exists())
print("DSSAT_HOME:", os.environ.get("DSSAT_HOME"))

try:
    dssat = dl.connect(interactive=False)
    print("Already available (not a fresh runtime):", dssat)
except dl.DSSATNotFoundError as e:
    print("Expected on a fresh runtime:\n", e)

## 3. Error messages for bad input

In [ ]:
for kwargs in ({"executable": "/no/such/dscsm048"}, {"path": "/tmp"}):
    try:
        dl.connect(interactive=False, **kwargs)
    except dl.DSSATNotFoundError as e:
        print(kwargs, "->\n ", e, "\n")

In [ ]:
# A file that exists and is executable but has the wrong name must be rejected too.
import pathlib
fake = pathlib.Path("/tmp/not_dssat")
fake.write_text("#!/bin/sh\n")
fake.chmod(0o755)
try:
    dl.connect(executable=fake, interactive=False)
except dl.DSSATNotFoundError as e:
    print(e)

## 4. Build DSSAT with `install()`

Build tools are checked first. Missing tools are **reported, never installed** (no `sudo`).

In [ ]:
import shutil
for tool in ("git", "cmake", "gfortran"):
    print(f"{tool:9}", shutil.which(tool) or "MISSING")

In [ ]:
# Only needed if a tool above says MISSING (Colab normally has all three).
# dssatlab will not do this for you, by design.
# !apt-get install -y git cmake gfortran

In [ ]:
# Resolves the latest stable GitHub release, then runs:
#   git clone --depth 1 --branch vX.Y.Z.W ...  ->  cmake -S -B ...  ->  cmake --build --parallel
# Takes several minutes; subprocess output is captured (shown only on failure).
import time
t0 = time.time()
dssat = dl.install()
print(dssat)
print(f"source={dssat.source!r} root={dssat.root}")
print(f"took {time.time() - t0:.0f}s")

## 5. What was written to disk

In [ ]:
from dssatlab import installer

print("cache root:", installer.cache_root())
!ls -la {installer.cache_root()}/*

In [ ]:
manifest = installer.cache_root() / dssat.version / "manifest.json"
print(manifest)
print(manifest.read_text())

In [ ]:
print(config.config_file())
print(config.config_file().read_text())

In [ ]:
# The executable exists and has the execute bit.
!ls -l {dssat.executable}
assert os.access(dssat.executable, os.X_OK)

## 6. Discovery after install

Precedence is **saved config > `DSSAT_HOME` > `PATH` > managed cache**. We check each layer by removing the ones above it.

In [ ]:
info = dl.detect()
print(info)
assert info.dssat_executable == dssat.executable

In [ ]:
# Layer 1: saved config
c = dl.connect(interactive=False)
print("with config        ->", c.source)
assert c.source == "config"

In [ ]:
# Layer 2: DSSAT_HOME (config removed). Accepts a directory holding the executable.
config.config_file().unlink()
os.environ["DSSAT_HOME"] = str(dssat.executable.parent)
c = dl.connect(interactive=False)
print("with DSSAT_HOME    ->", c.source)
assert c.source == "env"

In [ ]:
# Layer 3: PATH
config.config_file().unlink()
del os.environ["DSSAT_HOME"]
bin_dir = str(dssat.executable.parent)
os.environ["PATH"] = bin_dir + os.pathsep + os.environ["PATH"]
c = dl.connect(interactive=False)
print("with PATH          ->", c.source)
assert c.source == "default"

In [ ]:
# Layer 4: managed cache (no config, no env var, not on PATH)
config.config_file().unlink()
os.environ["PATH"] = os.pathsep.join(p for p in os.environ["PATH"].split(os.pathsep) if p != bin_dir)
c = dl.connect(interactive=False)
print("with managed cache ->", c.source, c.version)
assert c.source == "managed"

In [ ]:
# connect() saves what it found, so the next call is served from config again.
print(dl.connect(interactive=False).source)

## 7. Cache reuse

A second `install()` must find the manifest and skip clone and build.

In [ ]:
t0 = time.time()
again = dl.install()
elapsed = time.time() - t0
print(again, f"({elapsed:.1f}s)")
assert again.executable == dssat.executable
assert elapsed < 30, "expected a cache hit, not a rebuild"

In [ ]:
# An explicit version is also a cache hit when it matches the resolved latest.
print(dl.install(dssat.version))

## 8. Explicit connection

In [ ]:
by_exe = dl.connect(executable=dssat.executable, interactive=False)
by_dir = dl.connect(path=dssat.executable.parent, interactive=False)
print(by_exe)
print(by_dir)
assert by_exe.source == by_dir.source == "explicit"
assert by_exe.executable == by_dir.executable

In [ ]:
# Smoke test: launch the binary once (no experiment files, so it will complain or wait).
# We only check that it starts, i.e. it is a real, linkable executable.
import subprocess
try:
    r = subprocess.run([str(dssat.executable)], capture_output=True, text=True,
                       stdin=subprocess.DEVNULL, timeout=20)
    print("exit code:", r.returncode)
    print((r.stdout + r.stderr)[-600:])
except subprocess.TimeoutExpired:
    print("Started and was still running after 20s: it launched fine.")

## 9. Failure path: a version that does not exist

`install()` should raise `DSSATInstallError` from the failed `git clone`, showing the command and the tail of its output.

In [ ]:
try:
    dl.install("0.0.0.0")
except dl.DSSATInstallError as e:
    print(e)

In [ ]:
# The previous good connection must be unaffected.
print(dl.connect(interactive=False))

## 10. Reset

Removes only the saved config so you can re-run from step 2. The managed build stays in the cache;
set `WIPE_CACHE = True` to delete it too for a true cold start.

In [ ]:
WIPE_CACHE = False

if config.config_file().exists():
    config.config_file().unlink()
if WIPE_CACHE:
    shutil.rmtree(installer.cache_root(), ignore_errors=True)
print("config exists:", config.config_file().exists())
print("cache exists: ", installer.cache_root().exists())